# YOLO26 Baseline Benchmarks

Benchmark every baseline YOLO26 segmentation run on the held-out `ds-b-custom-sam3-v1/test` split using Ultralytics benchmark mode.

This notebook benchmarks `weights/best.pt` for each run, limits benchmark mode to PyTorch (`format="-"`), and writes summary outputs under `results/baseline_yolo26_twcc/benchmarks`.

In [ ]:
from __future__ import annotations

import json
import os
from contextlib import contextmanager
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print


def resolve_project_paths() -> tuple[Path, Path, Path]:
    cwd = Path.cwd().resolve()
    if cwd.name == "rebar-segementation-yolo26" and (cwd / "train.py").is_file():
        rebar_dir = cwd
        model_training_dir = rebar_dir.parent
        return model_training_dir.parent, model_training_dir, rebar_dir

    for candidate in (cwd, *cwd.parents):
        rebar_dir = candidate / "model-training" / "rebar-segementation-yolo26"
        if (rebar_dir / "train.py").is_file():
            return candidate, candidate / "model-training", rebar_dir

    raise FileNotFoundError("Could not locate model-training/rebar-segementation-yolo26 from the current working directory.")


REPO_ROOT, MODEL_TRAINING_DIR, REBAR_DIR = resolve_project_paths()
RUNS_DIR = REBAR_DIR / "results" / "baseline_yolo26_twcc" / "runs" / "segment"
RESULTS_DIR = REBAR_DIR / "results" / "baseline_yolo26_twcc" / "benchmarks"
DATASET_DIR = MODEL_TRAINING_DIR / "datasets" / "ds-b-custom-sam3-v1"
SOURCE_DATA_YAML = DATASET_DIR / "data.yaml"
BENCHMARK_DATA_YAML = RESULTS_DIR / "ds_b_test_benchmark_data.yaml"

WEIGHT_NAME = "best.pt"
IMGSZ = 640
HALF = False
FORMAT = "-"  # Ultralytics export format argument for native PyTorch.


def resolve_device(requested: str = "auto") -> str:
    if requested != "auto":
        return requested

    try:
        import torch
    except ImportError:
        return "cpu"

    if torch.cuda.is_available():
        return "0"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


DEVICE = resolve_device("auto")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {REPO_ROOT}")
print(f"Runs dir: {RUNS_DIR}")
print(f"Dataset dir: {DATASET_DIR}")
print(f"Benchmark outputs: {RESULTS_DIR}")
print(f"Device: {DEVICE}")

## Validate the Held-Out Test Split

Ultralytics `benchmark()` validates against the dataset YAML's `val` entry. To guarantee the held-out test split is used, this notebook writes a benchmark-only YAML where `val` points to `test/images`.

In [ ]:
IMAGE_EXTENSIONS = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}


def iter_images(images_dir: Path) -> list[Path]:
    return sorted(
        path for path in images_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def validate_yolo_split(dataset_dir: Path, split: str) -> dict[str, Any]:
    images_dir = dataset_dir / split / "images"
    labels_dir = dataset_dir / split / "labels"

    if not images_dir.is_dir():
        raise FileNotFoundError(f"Missing images directory: {images_dir}")
    if not labels_dir.is_dir():
        raise FileNotFoundError(f"Missing labels directory: {labels_dir}")

    images = iter_images(images_dir)
    image_stems = {path.stem for path in images}
    label_stems = {path.stem for path in labels_dir.glob("*.txt")}
    missing_labels = sorted(image_stems - label_stems)
    orphan_labels = sorted(label_stems - image_stems)

    if missing_labels:
        raise ValueError(f"{split} has images without labels: {missing_labels[:5]}")
    if orphan_labels:
        raise ValueError(f"{split} has labels without images: {orphan_labels[:5]}")

    return {
        "split": split,
        "images": len(images),
        "labels": len(label_stems),
        "images_dir": str(images_dir),
        "labels_dir": str(labels_dir),
    }


test_summary = validate_yolo_split(DATASET_DIR, "test")
assert SOURCE_DATA_YAML.is_file(), f"Missing source dataset YAML: {SOURCE_DATA_YAML}"
display(pd.DataFrame([test_summary]))

In [ ]:
def write_benchmark_data_yaml(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    yaml_text = "\n".join(
        [
            f"path: {DATASET_DIR.as_posix()}",
            "train: train/images",
            "val: test/images",
            "test: test/images",
            "",
            "nc: 1",
            "names:",
            "  0: rebar",
            "",
        ]
    )
    path.write_text(yaml_text, encoding="utf-8")


write_benchmark_data_yaml(BENCHMARK_DATA_YAML)
print(BENCHMARK_DATA_YAML.read_text(encoding="utf-8"))

## Discover Baseline Checkpoints

In [ ]:
def discover_checkpoints(runs_dir: Path, weight_name: str) -> pd.DataFrame:
    if not runs_dir.is_dir():
        raise FileNotFoundError(f"Missing runs directory: {runs_dir}")

    rows = []
    missing = []
    for run_dir in sorted(path for path in runs_dir.iterdir() if path.is_dir()):
        checkpoint = run_dir / "weights" / weight_name
        if checkpoint.is_file():
            rows.append(
                {
                    "run_name": run_dir.name,
                    "checkpoint": str(checkpoint),
                    "checkpoint_size_mb": round(checkpoint.stat().st_size / 1_000_000, 2),
                }
            )
        else:
            missing.append(str(checkpoint))

    if missing:
        raise FileNotFoundError("Some runs are missing the requested checkpoint:\n" + "\n".join(missing))
    if not rows:
        raise FileNotFoundError(f"No {weight_name} checkpoints found in {runs_dir}")

    return pd.DataFrame(rows)


checkpoints = discover_checkpoints(RUNS_DIR, WEIGHT_NAME)
print(f"Found {len(checkpoints)} checkpoints.")
display(checkpoints)

## Run PyTorch Benchmarks

The benchmark loop writes one subdirectory per model so each model keeps its own `benchmarks.log`. The summary CSV and JSON files are written after every row, so partial results are preserved if a long run is interrupted.

In [ ]:
@contextmanager
def pushd(path: Path):
    old_cwd = Path.cwd()
    path.mkdir(parents=True, exist_ok=True)
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(old_cwd)


def dataframe_from_benchmark(result: Any) -> pd.DataFrame:
    if hasattr(result, "to_dicts"):
        return pd.DataFrame(result.to_dicts())
    if hasattr(result, "to_pandas"):
        return result.to_pandas()
    if isinstance(result, pd.DataFrame):
        return result.copy()
    return pd.DataFrame(result)


def clean_value(value: Any) -> Any:
    if value is None:
        return None
    if isinstance(value, str):
        stripped = value.strip()
        if stripped in {"", "-", "None", "nan"}:
            return None
        try:
            return float(stripped)
        except ValueError:
            return stripped
    return value


def pick_column(columns: list[str], fragments: tuple[str, ...]) -> str | None:
    lowered = {column: column.lower() for column in columns}
    for column, lower in lowered.items():
        if all(fragment.lower() in lower for fragment in fragments):
            return column
    return None


def summarize_benchmark_frame(frame: pd.DataFrame) -> dict[str, Any]:
    if frame.empty:
        return {}

    columns = [str(column) for column in frame.columns]
    frame = frame.copy()
    frame.columns = columns

    format_col = pick_column(columns, ("format",))
    status_col = pick_column(columns, ("status",))
    speed_col = pick_column(columns, ("inference", "time"))
    fps_col = pick_column(columns, ("fps",))
    metric_col = next((column for column in columns if "map50-95" in column.lower()), None)
    size_col = pick_column(columns, ("size", "mb"))

    row = frame.iloc[0].to_dict()
    return {
        "benchmark_format": row.get(format_col) if format_col else None,
        "benchmark_status": row.get(status_col) if status_col else None,
        "map50_95": clean_value(row.get(metric_col)) if metric_col else None,
        "metric_column": metric_col,
        "inference_ms_per_image": clean_value(row.get(speed_col)) if speed_col else None,
        "fps": clean_value(row.get(fps_col)) if fps_col else None,
        "benchmark_size_mb": clean_value(row.get(size_col)) if size_col else None,
        "raw_columns": json.dumps(columns),
    }


def run_one_benchmark(run_name: str, checkpoint: str) -> tuple[dict[str, Any], pd.DataFrame]:
    from ultralytics.utils.benchmarks import benchmark

    model_output_dir = RESULTS_DIR / run_name
    started_at = datetime.now().isoformat(timespec="seconds")

    with pushd(model_output_dir):
        result = benchmark(
            model=checkpoint,
            data=str(BENCHMARK_DATA_YAML),
            imgsz=IMGSZ,
            half=HALF,
            device=DEVICE,
            verbose=False,
            format=FORMAT,
        )

    frame = dataframe_from_benchmark(result)
    summary = summarize_benchmark_frame(frame)
    summary.update(
        {
            "run_name": run_name,
            "checkpoint": checkpoint,
            "weight_name": WEIGHT_NAME,
            "data_yaml": str(BENCHMARK_DATA_YAML),
            "source_data_yaml": str(SOURCE_DATA_YAML),
            "dataset_split": "test via benchmark YAML val=test/images",
            "imgsz": IMGSZ,
            "half": HALF,
            "device": DEVICE,
            "format_arg": FORMAT,
            "output_dir": str(model_output_dir),
            "started_at": started_at,
            "finished_at": datetime.now().isoformat(timespec="seconds"),
            "error": None,
        }
    )
    return summary, frame


In [ ]:
SUMMARY_CSV = RESULTS_DIR / "baseline_yolo26_best_pt_pytorch_ds_b_test.csv"
SUMMARY_JSON = RESULTS_DIR / "baseline_yolo26_best_pt_pytorch_ds_b_test.json"
RAW_JSON = RESULTS_DIR / "baseline_yolo26_best_pt_pytorch_ds_b_test_raw.json"

summary_rows: list[dict[str, Any]] = []
raw_rows: list[dict[str, Any]] = []

for index, item in checkpoints.iterrows():
    run_name = item["run_name"]
    checkpoint = item["checkpoint"]
    print(f"[{index + 1}/{len(checkpoints)}] Benchmarking {run_name}")

    try:
        summary, raw_frame = run_one_benchmark(run_name, checkpoint)
    except Exception as exc:
        summary = {
            "run_name": run_name,
            "checkpoint": checkpoint,
            "weight_name": WEIGHT_NAME,
            "data_yaml": str(BENCHMARK_DATA_YAML),
            "source_data_yaml": str(SOURCE_DATA_YAML),
            "dataset_split": "test via benchmark YAML val=test/images",
            "imgsz": IMGSZ,
            "half": HALF,
            "device": DEVICE,
            "format_arg": FORMAT,
            "output_dir": str(RESULTS_DIR / run_name),
            "benchmark_format": None,
            "benchmark_status": "failed",
            "map50_95": None,
            "metric_column": None,
            "inference_ms_per_image": None,
            "fps": None,
            "benchmark_size_mb": None,
            "raw_columns": None,
            "started_at": None,
            "finished_at": datetime.now().isoformat(timespec="seconds"),
            "error": repr(exc),
        }
        raw_frame = pd.DataFrame()
        print(f"  failed: {exc!r}")

    summary_rows.append(summary)
    if not raw_frame.empty:
        raw = raw_frame.copy()
        raw.insert(0, "run_name", run_name)
        raw_rows.extend(raw.to_dict(orient="records"))

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(SUMMARY_CSV, index=False)
    SUMMARY_JSON.write_text(json.dumps(summary_rows, indent=2), encoding="utf-8")
    RAW_JSON.write_text(json.dumps(raw_rows, indent=2, default=str), encoding="utf-8")

summary_df = pd.DataFrame(summary_rows).sort_values(
    by=["map50_95", "inference_ms_per_image"],
    ascending=[False, True],
    na_position="last",
)
summary_df.to_csv(SUMMARY_CSV, index=False)
SUMMARY_JSON.write_text(json.dumps(summary_df.to_dict(orient="records"), indent=2), encoding="utf-8")

print(f"Wrote summary CSV: {SUMMARY_CSV}")
print(f"Wrote summary JSON: {SUMMARY_JSON}")
print(f"Wrote raw JSON: {RAW_JSON}")
display(summary_df)